In [ ]:
# =========================
# 0) Install & Imports
# =========================
!pip install -q --no-input transformers datasets accelerate peft bitsandbytes scikit-learn rapidfuzz matplotlib

import os, json, re, random
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

from datasets import Dataset
from collections import defaultdict
from typing import Dict, Any

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)

from peft import (
    LoraConfig,
    TaskType,
    prepare_model_for_kbit_training,
    get_peft_model,
)

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, roc_auc_score
)
from sklearn.model_selection import train_test_split as sk_train_test_split
import torch.nn.functional as F

from rapidfuzz.fuzz import ratio as fuzz_ratio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21

In [ ]:
# =========================
# 1) Reproducibility
# =========================
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); set_seed(SEED)



In [ ]:
# =========================
# 2) Paths & Model choice
# =========================
DATA_DIR = "/content/drive/MyDrive/NEW/Dataset"

# ORIGINAL sources (split happens from these)
ORIG_FAKE_PATH      = f"{DATA_DIR}/fake.csv"
ORIG_NONFAKE_PATH   = f"{DATA_DIR}/non-fake.csv"

# AUG pools (OPTIONAL; used only for TRAIN)
AUG_JSON_PATH       = f"{DATA_DIR}/balanced_augmented_dataset.json"    # optional
AUG_FAKE_CSV        = f"{DATA_DIR}/balanced_fake.csv"                  # optional
AUG_NONFAKE_CSV     = f"{DATA_DIR}/balanced_non_fake.csv"              # optional

# Optional external hold-out file (Review,Label)
HOLDOUT_CSV         = f"{DATA_DIR}/holdout.csv"                         # optional

MODEL_SAVE_PATH = "/content/drive/MyDrive/NEW/BanglaLLM/Bangla-s1k-llama-3.2-3B-Instruct"
MODEL_NAME      = "BanglaLLM/Bangla-s1k-llama-3.2-3B-Instruct"

os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

# Column names
TEXT_COL = "Review"

# Near-duplicate fuzzy threshold (tweak to 92/90 for stricter)
DEDUP_FUZZ_THR = 95


In [ ]:
# =========================
# 3) Load ORIGINALS → clean → dedup → stratified split
# =========================
def normalize_text(s: str) -> str:
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)
    return s

def _load_labeled_csv(path, label_value):
    df = pd.read_csv(path)
    assert TEXT_COL in df.columns, f"{path} must contain '{TEXT_COL}'"
    df = df[[TEXT_COL]].copy()
    df["Label"] = int(label_value)
    return df

# Load originals
df_fake     = _load_labeled_csv(ORIG_FAKE_PATH, 0)
df_nonfake  = _load_labeled_csv(ORIG_NONFAKE_PATH, 1)
df_orig = pd.concat([df_fake, df_nonfake], ignore_index=True)
print(f"📥 ORIGINAL loaded: {len(df_orig)} rows (fake={len(df_fake)}, non-fake={len(df_nonfake)})")

# Clean + filter
df_orig[TEXT_COL] = df_orig[TEXT_COL].astype(str).map(normalize_text)
df_orig = df_orig[(df_orig[TEXT_COL] != "") & df_orig["Label"].isin([0,1])].copy()

# Dedup originals with majority label if conflict
counts = defaultdict(lambda: [0,0])  # [count_0, count_1]
for _, r in df_orig.iterrows():
    counts[r[TEXT_COL].lower()][r["Label"]] += 1

seen, rows, conflicts = set(), [], 0
for _, r in df_orig.iterrows():
    k = r[TEXT_COL].lower()
    if k in seen: continue
    seen.add(k)
    c0, c1 = counts[k]
    if c0>0 and c1>0: conflicts += 1
    maj = r["Label"] if c0==c1 else (0 if c0>c1 else 1)
    rows.append({TEXT_COL: r[TEXT_COL], "Label": maj})

df_orig_dedup = pd.DataFrame(rows)
print(f"🧹 ORIGINAL dedup: {len(df_orig_dedup)} unique (removed {len(df_orig)-len(df_orig_dedup)}), conflicts: {conflicts}")

# Stratified split (test untouched by augmentation)
idx = np.arange(len(df_orig_dedup))
tr_idx, te_idx = sk_train_test_split(
    idx, test_size=0.20, random_state=SEED, stratify=df_orig_dedup["Label"]
)
train_df = df_orig_dedup.iloc[tr_idx].reset_index(drop=True)
test_df  = df_orig_dedup.iloc[te_idx].reset_index(drop=True)
print(f"✅ ORIGINAL split → Train: {len(train_df)} | Test: {len(test_df)}")



📥 ORIGINAL loaded: 13586 rows (fake=5876, non-fake=7710)
🧹 ORIGINAL dedup: 13424 unique (removed 162), conflicts: 0
✅ ORIGINAL split → Train: 10739 | Test: 2685


In [ ]:
# =========================
# 3b) AUGMENT → only to TRAIN (filter overlaps with TEST)
# =========================
def load_aug_pool():
    pool = []
    if os.path.isfile(AUG_JSON_PATH):
        with open(AUG_JSON_PATH, "r", encoding="utf-8") as f:
            js = json.load(f)
        for it in js:
            txt = normalize_text(it.get("Review", ""))
            lab = it.get("Label", None)
            if txt and lab in (0,1):
                pool.append({TEXT_COL: txt, "Label": int(lab)})
        print(f"➕ AUG from JSON: {len(pool)}")
    else:
        def _load_aug_csv(pth, lab):
            if os.path.isfile(pth):
                df = pd.read_csv(pth)
                assert TEXT_COL in df.columns, f"{pth} must contain '{TEXT_COL}'"
                here = (
                    df[[TEXT_COL]].assign(Label=int(lab))
                    .dropna()
                    .astype({TEXT_COL: str})
                )
                here[TEXT_COL] = here[TEXT_COL].map(normalize_text)
                return here.to_dict("records")
            return []
        pool_fake = _load_aug_csv(AUG_FAKE_CSV, 0)
        pool_nonf = _load_aug_csv(AUG_NONFAKE_CSV, 1)
        pool = pool_fake + pool_nonf
        print(f"➕ AUG from CSVs: {len(pool)} (fake={len(pool_fake)}, nonfake={len(pool_nonf)})")
    return pool

aug_pool = load_aug_pool()

def is_near_duplicate(s, bank, thr=DEDUP_FUZZ_THR):
    s_norm = s.lower()
    if s_norm in bank:
        return True
    if thr is None:
        return False
    for t in bank:
        if fuzz_ratio(s_norm, t) >= thr:
            return True
    return False

# Filter aug against TEST (exact + near-dup)
test_bank = test_df[TEXT_COL].str.lower().tolist()
aug_filtered = []
if aug_pool:
    for it in aug_pool:
        txt = it[TEXT_COL]; lab = it["Label"]
        if not txt or lab not in (0,1): continue
        if is_near_duplicate(txt, test_bank, thr=DEDUP_FUZZ_THR): continue
        aug_filtered.append({TEXT_COL: txt, "Label": lab})
print(f"✅ AUG after test-filter: {len(aug_filtered)}")

# Merge and final train dedup
if aug_filtered:
    train_merge = pd.concat([train_df, pd.DataFrame(aug_filtered)], ignore_index=True)
else:
    train_merge = train_df.copy()

train_merge["__k"] = train_merge[TEXT_COL].str.lower().str.strip()
train_merge = train_merge.drop_duplicates(subset="__k").drop(columns="__k")
print(f"📦 Final TRAIN size (orig + aug, deduped): {len(train_merge)}")

# HF datasets
train_raw = Dataset.from_pandas(train_merge.reset_index(drop=True))
test_raw  = Dataset.from_pandas(test_df.reset_index(drop=True))

# Keep copies for later reporting
test_texts = test_raw[TEXT_COL]
test_labels = test_raw["Label"]



➕ AUG from JSON: 12400
✅ AUG after test-filter: 9852
📦 Final TRAIN size (orig + aug, deduped): 11011


In [ ]:
# =========================
# 4) Tokenizer (pad fix)
# =========================
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, trust_remote_code=True)
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False, trust_remote_code=True)

tokenizer.padding_side = "right"

ADDED_PAD_TOKEN = False
if tokenizer.pad_token_id is None:
    if tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token
    else:
        tokenizer.add_special_tokens({"pad_token": "<pad>"})
        ADDED_PAD_TOKEN = True

PAD_TOKEN_ID = tokenizer.pad_token_id
assert PAD_TOKEN_ID is not None

MAX_LEN = 256

def tok_fn(examples):
    toks = tokenizer(
        examples[TEXT_COL],
        truncation=True,
        max_length=MAX_LEN,
        padding=False,
        return_attention_mask=True,
    )
    toks["labels"] = [int(x) for x in examples["Label"]]
    return toks

train_ds = train_raw.map(tok_fn, batched=True, remove_columns=train_raw.column_names)
test_ds  = test_raw.map(tok_fn,  batched=True, remove_columns=test_raw.column_names)


# =========================
# 5) 4-bit + LoRA setup
# =========================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# pad sync + optional resize
model.config.pad_token_id = PAD_TOKEN_ID
if ADDED_PAD_TOKEN:
    model.resize_token_embeddings(len(tokenizer))

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)
model = get_peft_model(model, lora_cfg)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

model.print_trainable_parameters()


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/343 [00:00<?, ?B/s]

Map:   0%|          | 0/11011 [00:00<?, ? examples/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at BanglaLLM/Bangla-s1k-llama-3.2-3B-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 4,593,664 || all params: 3,217,349,632 || trainable%: 0.1428


In [ ]:
# =========================
# 6) Data collator & metrics
# =========================
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    pr, rc, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    return {"accuracy": acc, "precision": pr, "recall": rc, "f1": f1}


In [ ]:
# =========================
# 7) TrainingArguments & Trainer
# =========================
training_args = TrainingArguments(
    output_dir=MODEL_SAVE_PATH,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    save_strategy="epoch",
    logging_steps=50,
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)



/tmp/ipython-input-2781989432.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# =========================
# 8) Resume from checkpoint (Drive)
# =========================
resume_ckpt = None
if os.path.isdir(MODEL_SAVE_PATH):
    cks = [d for d in os.listdir(MODEL_SAVE_PATH) if d.startswith("checkpoint-")]
    if cks:
        cks_sorted = sorted(cks, key=lambda x: int(x.split("-")[-1]))
        resume_ckpt = os.path.join(MODEL_SAVE_PATH, cks_sorted[-1])
        print(f"🔁 Resuming from: {resume_ckpt}")
    else:
        print("🆕 No checkpoint found; training from scratch.")
else:
    print("🆕 Output dir not found; will create.")


🔁 Resuming from: /content/drive/MyDrive/NEW/BanglaLLM/Bangla-s1k-llama-3.2-3B-Instruct/checkpoint-690


In [ ]:
# =========================
# 9) Train
# =========================
trainer.train(resume_from_checkpoint=resume_ckpt)

# Save a final copy
final_dir = os.path.join(MODEL_SAVE_PATH, "final")
os.makedirs(final_dir, exist_ok=True)
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print(f"✅ Final model saved to: {final_dir}")


Step,Training Loss
700,0.504700
750,0.512700
800,0.641200
850,0.679000
900,0.783900
950,0.553000
1000,0.719800


✅ Final model saved to: /content/drive/MyDrive/NEW/BanglaLLM/Bangla-s1k-llama-3.2-3B-Instruct/final


In [ ]:
# =========================
# 10) Evaluate + Save predictions & probabilities
# =========================
eval_res = trainer.evaluate()
print("📊 Eval (HF):", eval_res)

pred_out = trainer.predict(test_ds)
logits = pred_out.predictions
y_true = np.array(test_labels, dtype=int)
y_pred = np.argmax(logits, axis=1)
probs  = F.softmax(torch.tensor(logits), dim=1).cpu().numpy()

pred_df = pd.DataFrame({
    "Review": test_texts,
    "True": y_true,
    "Pred": y_pred,
    "Prob_Fake(0)": probs[:, 0],
    "Prob_NonFake(1)": probs[:, 1],
})
pred_csv = os.path.join(MODEL_SAVE_PATH, "test_predictions_with_probs.csv")
pred_df.to_csv(pred_csv, index=False, encoding="utf-8")
print(f"✅ Saved predictions: {pred_csv}")

report_str = classification_report(
    y_true, y_pred, labels=[0,1],
    target_names=["Fake(0)", "Non-Fake(1)"], digits=4, zero_division=0
)
print("\n🔎 Classification Report:\n", report_str)
with open(os.path.join(MODEL_SAVE_PATH, "classification_report.txt"), "w", encoding="utf-8") as f:
    f.write(report_str)

metrics_json = {
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "precision_weighted": float(precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)[0]),
    "recall_weighted": float(precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)[1]),
    "f1_weighted": float(precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)[2]),
}
with open(os.path.join(MODEL_SAVE_PATH, "metrics.json"), "w", encoding="utf-8") as f:
    json.dump(metrics_json, f, ensure_ascii=False, indent=2)
print("✅ Metrics saved.")

# Misclassified cases
mis_df = pred_df[pred_df["True"] != pred_df["Pred"]].copy()
mis_csv = os.path.join(MODEL_SAVE_PATH, "misclassified_cases.csv")
mis_df.to_csv(mis_csv, index=False, encoding="utf-8")
print(f"🧪 Misclassified cases saved: {mis_csv}")

# Per-class + CM + splits
report_dict = classification_report(
    y_true, y_pred, labels=[0,1],
    target_names=["Fake(0)", "Non-Fake(1)"], zero_division=0, output_dict=True, digits=4
)
pd.DataFrame(report_dict).transpose().to_csv(os.path.join(MODEL_SAVE_PATH, "per_class_metrics.csv"),
                                             index=True, encoding="utf-8")
cm = confusion_matrix(y_true, y_pred, labels=[0,1])
cm_df = pd.DataFrame(cm, index=["True_Fake(0)", "True_NonFake(1)"], columns=["Pred_Fake(0)", "Pred_NonFake(1)"])
cm_df.to_csv(os.path.join(MODEL_SAVE_PATH, "confusion_matrix.csv"), encoding="utf-8")
pred_df[pred_df["True"] == 0].to_csv(os.path.join(MODEL_SAVE_PATH, "GT_Fake_only.csv"), index=False, encoding="utf-8")
pred_df[pred_df["True"] == 1].to_csv(os.path.join(MODEL_SAVE_PATH, "GT_NonFake_only.csv"), index=False, encoding="utf-8")
pred_df[pred_df["Pred"] == 0].to_csv(os.path.join(MODEL_SAVE_PATH, "PRED_Fake_only.csv"), index=False, encoding="utf-8")
pred_df[pred_df["Pred"] == 1].to_csv(os.path.join(MODEL_SAVE_PATH, "PRED_NonFake_only.csv"), index=False, encoding="utf-8")

try:
    auc_nonfake = roc_auc_score(y_true, probs[:, 1])
    auc_fake    = roc_auc_score(1 - y_true, probs[:, 0])
    with open(os.path.join(MODEL_SAVE_PATH, "auc_scores.json"), "w", encoding="utf-8") as f:
        json.dump({"AUC_NonFake(1)": float(auc_nonfake), "AUC_Fake(0)": float(auc_fake)}, f, indent=2, ensure_ascii=False)
    print(f"✅ AUC saved (NonFake: {auc_nonfake:.4f}, Fake: {auc_fake:.4f})")
except Exception as e:
    print("AUC computation skipped:", e)

# Threshold sweep for class 1
ths = np.linspace(0.30, 0.70, 9)
rows = []
for t in ths:
    y_hat = (probs[:,1] >= t).astype(int)
    acc = accuracy_score(y_true, y_hat)
    pr1, rc1, f1_1, _ = precision_recall_fscore_support(y_true, y_hat, average="binary", pos_label=1, zero_division=0)
    cm_t = confusion_matrix(y_true, y_hat, labels=[0,1])
    rows.append({"threshold": t, "acc": acc, "prec_1": pr1, "rec_1": rc1, "f1_1": f1_1,
                 "FN_nonfake": int(cm_t[1,0]), "FP_nonfake": int(cm_t[0,1])})
thr_df = pd.DataFrame(rows)
thr_df.to_csv(os.path.join(MODEL_SAVE_PATH, "threshold_tuning_nonfake.csv"), index=False, encoding="utf-8")
best = max(rows, key=lambda r: r["f1_1"])
best_t = best["threshold"]
print(f"⭐ Best threshold for class 1 (by F1): {best_t:.2f} | F1={best['f1_1']:.4f} | FN1={best['FN_nonfake']} | FP1={best['FP_nonfake']}")
pred_df_thr = pred_df.copy(); pred_df_thr["Pred_thr"] = (probs[:,1] >= best_t).astype(int)
pred_df_thr.to_csv(os.path.join(MODEL_SAVE_PATH, f"test_predictions_thr_{best_t:.2f}.csv"), index=False, encoding="utf-8")



📊 Eval (HF): {'eval_loss': 0.11771388351917267, 'eval_accuracy': 0.9716945996275606, 'eval_precision': 0.9721203256148535, 'eval_recall': 0.9716945996275606, 'eval_f1': 0.9716131346622271, 'eval_runtime': 264.9739, 'eval_samples_per_second': 10.133, 'eval_steps_per_second': 1.268, 'epoch': 3.0}
✅ Saved predictions: /content/drive/MyDrive/NEW/BanglaLLM/Bangla-s1k-llama-3.2-3B-Instruct/test_predictions_with_probs.csv

🔎 Classification Report:
               precision    recall  f1-score   support

     Fake(0)     0.9864    0.9468    0.9662      1146
 Non-Fake(1)     0.9615    0.9903    0.9757      1539

    accuracy                         0.9717      2685
   macro avg     0.9739    0.9685    0.9709      2685
weighted avg     0.9721    0.9717    0.9716      2685

✅ Metrics saved.
🧪 Misclassified cases saved: /content/drive/MyDrive/NEW/BanglaLLM/Bangla-s1k-llama-3.2-3B-Instruct/misclassified_cases.csv
✅ AUC saved (NonFake: 0.9927, Fake: 0.9927)
⭐ Best threshold for class 1 (by F1): 0.65 

In [ ]:
# =========================
# 11) Hold-out external evaluation (optional)
# =========================
def _load_holdout_csv(path):
    df = pd.read_csv(path)
    assert TEXT_COL in df.columns and "Label" in df.columns, f"{path} needs '{TEXT_COL}' and 'Label'"
    df = df[[TEXT_COL, "Label"]].dropna()
    df[TEXT_COL] = df[TEXT_COL].astype(str).map(normalize_text)
    df = df[(df[TEXT_COL]!="") & df["Label"].isin([0,1])]
    return df

if os.path.isfile(HOLDOUT_CSV):
    hold_df = _load_holdout_csv(HOLDOUT_CSV)
    def tok_hold(batch):
        out = tokenizer(batch[TEXT_COL], truncation=True, max_length=MAX_LEN, padding=False, return_attention_mask=True)
        out["labels"] = batch["Label"]
        return out
    hold_ds = Dataset.from_pandas(hold_df.reset_index(drop=True)).map(
        tok_hold, batched=True, remove_columns=hold_df.columns.tolist()
    )
    hold_out = trainer.predict(hold_ds)
    hold_logits = hold_out.predictions
    hold_probs  = F.softmax(torch.tensor(hold_logits), dim=1).cpu().numpy()
    hold_true   = hold_df["Label"].to_numpy(dtype=int)
    hold_pred   = np.argmax(hold_logits, axis=1)

    hold_report = classification_report(hold_true, hold_pred, labels=[0,1],
                                        target_names=["Fake(0)", "Non-Fake(1)"], digits=4, zero_division=0)
    print("\n🛡️ Hold-out Report:\n", hold_report)
    with open(os.path.join(MODEL_SAVE_PATH, "holdout_report.txt"), "w", encoding="utf-8") as f:
        f.write(hold_report)

    pd.DataFrame({
        "Review": hold_df[TEXT_COL],
        "True": hold_true, "Pred": hold_pred,
        "Prob_Fake(0)": hold_probs[:,0], "Prob_NonFake(1)": hold_probs[:,1],
    }).to_csv(os.path.join(MODEL_SAVE_PATH, "holdout_predictions.csv"), index=False, encoding="utf-8")
    print("✅ Hold-out predictions saved.")
else:
    print("ℹ️ No hold-out CSV found; skipping external evaluation.")

In [ ]:
# =========================
# 12) Multi-seed experiment (fast sanity check on T4)
# =========================
MULTI_SEEDS = [42, 13, 21]   # add more if you like
EPOCHS_MULTI = 2             # keep small for T4; raise if time allows
BATCH_TRAIN_MULTI = 4        # drop to 2 if OOM

from copy import deepcopy

def prepare_data_for_seed(seed, dedup_thr=DEDUP_FUZZ_THR):
    # rebuild ORIGINAL split per seed
    df_fake = pd.read_csv(ORIG_FAKE_PATH)[[TEXT_COL]].assign(Label=0)
    df_nonf = pd.read_csv(ORIG_NONFAKE_PATH)[[TEXT_COL]].assign(Label=1)
    df_o = pd.concat([df_fake, df_nonf], ignore_index=True)
    df_o[TEXT_COL] = df_o[TEXT_COL].astype(str).map(normalize_text)
    df_o = df_o[(df_o[TEXT_COL]!="") & df_o["Label"].isin([0,1])]

    # original dedup
    cts = defaultdict(lambda: [0,0])
    for _, r in df_o.iterrows(): cts[r[TEXT_COL].lower()][r["Label"]] += 1
    seen, rows = set(), []
    for _, r in df_o.iterrows():
        k = r[TEXT_COL].lower()
        if k in seen: continue
        seen.add(k)
        c0, c1 = cts[k]
        maj = r["Label"] if c0==c1 else (0 if c0>c1 else 1)
        rows.append({TEXT_COL: r[TEXT_COL], "Label": maj})
    df_o = pd.DataFrame(rows)

    # stratified split
    idx = np.arange(len(df_o))
    tr_idx, te_idx = sk_train_test_split(idx, test_size=0.20, random_state=seed, stratify=df_o["Label"])
    tr_df = df_o.iloc[tr_idx].reset_index(drop=True)
    te_df = df_o.iloc[te_idx].reset_index(drop=True)

    # load aug pool
    aug_pool = []
    if os.path.isfile(AUG_JSON_PATH):
        with open(AUG_JSON_PATH, "r", encoding="utf-8") as f:
            js = json.load(f)
        for it in js:
            txt = normalize_text(it.get("Review","")); lab = it.get("Label", None)
            if txt and lab in (0,1): aug_pool.append({TEXT_COL: txt, "Label": int(lab)})
    else:
        def _load_aug_csv(pth, lab):
            if os.path.isfile(pth):
                df = pd.read_csv(pth)
                assert TEXT_COL in df.columns
                df = df[[TEXT_COL]].assign(Label=int(lab)).dropna()
                df[TEXT_COL] = df[TEXT_COL].astype(str).map(normalize_text)
                return df.to_dict("records")
            return []
        aug_pool = _load_aug_csv(AUG_FAKE_CSV, 0) + _load_aug_csv(AUG_NONFAKE_CSV, 1)

    # filter vs TEST (exact + near-dup)
    bank = te_df[TEXT_COL].str.lower().tolist()
    def _dup(x):
        if x.lower() in bank: return True
        if dedup_thr is None: return False
        for t in bank:
            if fuzz_ratio(x.lower(), t) >= dedup_thr: return True
        return False
    aug_filt = [it for it in aug_pool if not _dup(it[TEXT_COL])]

    # merge + final dedup
    tr_merge = pd.concat([tr_df, pd.DataFrame(aug_filt)], ignore_index=True)
    tr_merge["__k"] = tr_merge[TEXT_COL].str.lower().str.strip()
    tr_merge = tr_merge.drop_duplicates(subset="__k").drop(columns="__k")

    # to HF + tokenize
    tr_raw = Dataset.from_pandas(tr_merge.reset_index(drop=True))
    te_raw = Dataset.from_pandas(te_df.reset_index(drop=True))

    def _tok(batch):
        out = tokenizer(batch[TEXT_COL], truncation=True, max_length=MAX_LEN, padding=False, return_attention_mask=True)
        out["labels"] = batch["Label"]
        return out
    tr_ds = tr_raw.map(_tok, batched=True, remove_columns=tr_raw.column_names)
    te_ds = te_raw.map(_tok, batched=True, remove_columns=te_raw.column_names)
    return tr_ds, te_ds, te_raw[TEXT_COL], te_raw["Label"]

def train_one_seed(seed):
    print(f"\n==== SEED {seed} ====")
    tr_ds, te_ds, te_texts, te_labels = prepare_data_for_seed(seed, DEDUP_FUZZ_THR)

    model_s = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2, quantization_config=bnb_config,
        device_map="auto", trust_remote_code=True
    )
    model_s.config.pad_token_id = PAD_TOKEN_ID
    if ADDED_PAD_TOKEN: model_s.resize_token_embeddings(len(tokenizer))
    model_s.config.use_cache = False
    model_s = prepare_model_for_kbit_training(model_s)
    model_s = get_peft_model(model_s, lora_cfg)
    model_s.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

    ta = TrainingArguments(
        output_dir=os.path.join(MODEL_SAVE_PATH, f"ms_seed{seed}"),
        per_device_train_batch_size=BATCH_TRAIN_MULTI,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=8,
        num_train_epochs=EPOCHS_MULTI,
        learning_rate=2e-4,
        fp16=True,
        save_strategy="no",
        logging_steps=50,
        report_to="none",
        remove_unused_columns=False,
    )
    tr = Trainer(
        model=model_s, args=ta, train_dataset=tr_ds, eval_dataset=te_ds,
        tokenizer=tokenizer, data_collator=data_collator, compute_metrics=compute_metrics
    )

    tr.train()
    eval_res = tr.evaluate()
    print("Eval:", eval_res)

    pred = tr.predict(te_ds)
    logits = pred.predictions
    y_true = np.array(te_labels, dtype=int)
    y_pred = np.argmax(logits, axis=1)

    res = {
        "seed": seed,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_weighted": float(precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)[0]),
        "recall_weighted": float(precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)[1]),
        "f1_weighted": float(precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)[2]),
    }

    # learning-curve logs
    df_hist = pd.DataFrame(tr.state.log_history)
    keep_cols = [c for c in ["epoch", "loss", "eval_loss"] if c in df_hist.columns]
    if keep_cols:
        df_hist[keep_cols].to_csv(os.path.join(MODEL_SAVE_PATH, f"ms_seed{seed}_log.csv"), index=False)
        try:
            plt.figure()
            if "loss" in df_hist.columns: plt.plot(df_hist.index, df_hist["loss"], label="train_loss")
            if "eval_loss" in df_hist.columns: plt.plot(df_hist.index, df_hist["eval_loss"], label="eval_loss")
            plt.title(f"Learning Curve (seed={seed})")
            plt.xlabel("log step"); plt.ylabel("loss"); plt.legend(); plt.tight_layout()
            plt.savefig(os.path.join(MODEL_SAVE_PATH, f"ms_seed{seed}_curve.png")); plt.close()
        except Exception as e:
            print("Plot skipped:", e)

    return res

multi_rows = []
for s in MULTI_SEEDS:
    try:
        multi_rows.append(train_one_seed(s))
    except RuntimeError as e:
        print(f"Seed {s} failed (OOM?). Tip: set BATCH_TRAIN_MULTI=2. Error:", e)

if multi_rows:
    df_ms = pd.DataFrame(multi_rows)
    df_ms.to_csv(os.path.join(MODEL_SAVE_PATH, "multi_seed_summary.csv"),
                 index=False, encoding="utf-8")
    mean_metrics = df_ms.drop(columns=["seed"]).mean().to_dict()
    std_metrics  = df_ms.drop(columns=["seed"]).std(ddof=0).to_dict()
    with open(os.path.join(MODEL_SAVE_PATH, "multi_seed_mean_std.json"), "w", encoding="utf-8") as f:
        json.dump({"mean": mean_metrics, "std": std_metrics}, f, indent=2, ensure_ascii=False)
    print("\n📊 Multi-seed mean:", mean_metrics)
    print("📏 Multi-seed std :", std_metrics)